# Verify grid sizes and memory footprint per archive variable

Two things, per archive variable (see `docs/model_design.md` §2.1-§2.3):
1. **Grid size** — confirm each variable's on-disk array shape actually matches what `core.config.VARIABLE_GRID` + `WAVE_GRID_SHAPE`/`WIND_GRID_SHAPE`/`OCEAN_GRID_SHAPE` claim, and that its on-disk dtype matches `core.io`'s continuous/direction classification — by round-tripping through the real `core.io` loaders on a synthetic archive, not just reading the constants back.
2. **Memory footprint** — per-snapshot and full-training-window (2022-01-01 to 2025-12-31 per `docs/model_design.md` §1) size, both raw on-disk (`float16`/`int8`) and decoded in-memory (`float64`, what `core.io.load_snapshot`/`train.ingest.load_training_window` actually produce) — broken down per variable, per grid, and for the tier-1 modeled variable subset specifically.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "src")))

import numpy as np
import pandas as pd
from datetime import datetime, timezone

from core.config import VARIABLE_GRID, WAVE_GRID_SHAPE, WIND_GRID_SHAPE, OCEAN_GRID_SHAPE
from core.io import _CONTINUOUS_VARIABLES, _DIRECTION_ORIGIN_CODES, enumerate_snapshots

GRID_SHAPES = {"wave": WAVE_GRID_SHAPE, "wind": WIND_GRID_SHAPE, "ocean": OCEAN_GRID_SHAPE}

# `_CONTINUOUS_VARIABLES`/`_DIRECTION_ORIGIN_CODES` are private to core.io -- reaching into
# them here is deliberate: this notebook exists specifically to check they haven't drifted
# apart from VARIABLE_GRID (e.g. a variable added to one but not the other).
assert set(_CONTINUOUS_VARIABLES) | set(_DIRECTION_ORIGIN_CODES) == set(VARIABLE_GRID), \
    "core.io's dtype classification no longer covers exactly core.config.VARIABLE_GRID's variables"
assert set(_CONTINUOUS_VARIABLES) & set(_DIRECTION_ORIGIN_CODES) == set(), \
    "a variable is classified as BOTH continuous and direction -- that's a real bug"
print(f"{len(VARIABLE_GRID)} variables, dtype-classified with no gaps or overlaps -- OK")

rows = []
for var, grid in VARIABLE_GRID.items():
    shape = GRID_SHAPES[grid]
    is_continuous = var in _CONTINUOUS_VARIABLES
    rows.append({
        "variable": var,
        "grid": grid,
        "grid_shape": shape,
        "n_cells": shape[0] * shape[1],
        "on_disk_dtype": "float16" if is_continuous else "int8",
        "on_disk_itemsize": 2 if is_continuous else 1,
    })
variable_table = pd.DataFrame(rows).set_index("variable")
variable_table


16 variables, dtype-classified with no gaps or overlaps -- OK


,grid,grid_shape,n_cells,on_disk_dtype,on_disk_itemsize
variable,,,,,
sigwh,wave,"(621, 1440)",894240,float16,2
wsh,wave,"(621, 1440)",894240,float16,2
wsp,wave,"(621, 1440)",894240,float16,2
wsd,wave,"(621, 1440)",894240,int8,1
pwh,wave,"(621, 1440)",894240,float16,2
pwp,wave,"(621, 1440)",894240,float16,2
pwd,wave,"(621, 1440)",894240,int8,1
pswh,wave,"(621, 1440)",894240,float16,2
pswp,wave,"(621, 1440)",894240,float16,2


## 1. Empirical verification: round-trip every variable through a synthetic archive

The table above is only as trustworthy as the constants it's built from. This section actually exercises `core.io.snapshot_path`/`load_raw`/`load_snapshot` against a real (synthetic) archive on disk, for every one of the 16 variables, and checks the loaded array's shape/dtype match the table — not just re-printing the same constants back.

In [2]:
import tempfile
from pathlib import Path
from core.io import snapshot_path, load_raw, load_snapshot

t0 = datetime(2024, 6, 15, 9, tzinfo=timezone.utc)

with tempfile.TemporaryDirectory() as tmp:
    data_dir = Path(tmp)

    # Write one synthetic snapshot per variable, shaped/dtyped per the table above.
    for var, grid in VARIABLE_GRID.items():
        shape = GRID_SHAPES[grid]
        path = snapshot_path(data_dir, t0, var)
        path.parent.mkdir(parents=True, exist_ok=True)
        dtype = np.float16 if var in _CONTINUOUS_VARIABLES else np.int8
        np.save(path, np.zeros(shape, dtype=dtype))

    # Read every variable back through the real loaders and check against the table.
    verify_rows = []
    for var, grid in VARIABLE_GRID.items():
        expected_shape = GRID_SHAPES[grid]
        expected_raw_dtype = np.float16 if var in _CONTINUOUS_VARIABLES else np.int8
        raw = load_raw(data_dir, t0, var)
        decoded = load_snapshot(data_dir, t0, var)
        ok = (
            raw.shape == expected_shape
            and raw.dtype == expected_raw_dtype
            and decoded.shape == expected_shape
            and decoded.dtype == np.float64
        )
        verify_rows.append({
            "variable": var, "grid": grid, "raw_shape": raw.shape, "raw_dtype": str(raw.dtype),
            "decoded_dtype": str(decoded.dtype), "matches_table": ok,
        })

verify_table = pd.DataFrame(verify_rows).set_index("variable")
print(f"all {len(verify_table)} variables match: {bool(verify_table['matches_table'].all())}")
verify_table


all 16 variables match: True


,grid,raw_shape,raw_dtype,decoded_dtype,matches_table
variable,,,,,
sigwh,wave,"(621, 1440)",float16,float64,True
wsh,wave,"(621, 1440)",float16,float64,True
wsp,wave,"(621, 1440)",float16,float64,True
wsd,wave,"(621, 1440)",int8,float64,True
pwh,wave,"(621, 1440)",float16,float64,True
pwp,wave,"(621, 1440)",float16,float64,True
pwd,wave,"(621, 1440)",int8,float64,True
pswh,wave,"(621, 1440)",float16,float64,True
pswp,wave,"(621, 1440)",float16,float64,True


## 2. Memory footprint

Per-snapshot bytes = `n_cells * itemsize`, both raw on-disk (`float16`/`int8`, as stored) and decoded in-memory (`float64`, what `core.io.load_snapshot` and `train.ingest.load_training_window` actually hand back). Full-archive figures use the real training window from `docs/model_design.md` §1 (2022-01-01 to 2025-12-31) and the real snapshot count from `core.io.enumerate_snapshots` — not a hardcoded "~11,700".

In [3]:
TRAIN_START = datetime(2022, 1, 1, tzinfo=timezone.utc)
TRAIN_END = datetime(2026, 1, 1, tzinfo=timezone.utc)  # enumerate_snapshots end is exclusive
n_snapshots = sum(1 for _ in enumerate_snapshots(TRAIN_START, TRAIN_END))
print(f"training window {TRAIN_START.date()} to 2025-12-31: {n_snapshots} snapshots at 3h cadence")

variable_table["raw_bytes_per_snapshot"] = variable_table["n_cells"] * variable_table["on_disk_itemsize"]
variable_table["decoded_bytes_per_snapshot"] = variable_table["n_cells"] * 8  # float64
variable_table["raw_GB_full_archive"] = variable_table["raw_bytes_per_snapshot"] * n_snapshots / 1e9
variable_table["decoded_GB_full_archive"] = variable_table["decoded_bytes_per_snapshot"] * n_snapshots / 1e9
variable_table[["grid", "raw_bytes_per_snapshot", "decoded_bytes_per_snapshot",
                "raw_GB_full_archive", "decoded_GB_full_archive"]].round(2)


training window 2022-01-01 to 2025-12-31: 11688 snapshots at 3h cadence


,grid,raw_bytes_per_snapshot,decoded_bytes_per_snapshot,raw_GB_full_archive,decoded_GB_full_archive
variable,,,,,
sigwh,wave,1788480,7153920,20.90,83.62
wsh,wave,1788480,7153920,20.90,83.62
wsp,wave,1788480,7153920,20.90,83.62
wsd,wave,894240,7153920,10.45,83.62
pwh,wave,1788480,7153920,20.90,83.62
pwp,wave,1788480,7153920,20.90,83.62
pwd,wave,894240,7153920,10.45,83.62
pswh,wave,1788480,7153920,20.90,83.62
pswp,wave,1788480,7153920,20.90,83.62


In [4]:
print("=== Per-grid totals, full training window, ALL 16 variables ===")
by_grid = variable_table.groupby("grid")[["raw_GB_full_archive", "decoded_GB_full_archive"]].sum()
print(by_grid.round(1))
print()
print(f"ALL VARIABLES, raw on-disk:      {variable_table['raw_GB_full_archive'].sum():.1f} GB")
print(f"ALL VARIABLES, decoded in-memory: {variable_table['decoded_GB_full_archive'].sum():.1f} GB "
      f"({variable_table['decoded_GB_full_archive'].sum()/1000:.2f} TB) "
      f"-- loading every variable decoded for the full window at once is not realistic RAM.")

# Tier-1: the variable set weather.model.WeatherModel actually trains on (see
# weather/model.py's pipeline construction, tasks/08-weather-model-rewrite.md) --
# NOT the model_design.md §8.6 "wad" example line, which is a known doc bug (tasks/README.md).
TIER1_VARIABLES = ["sigwh", "wsh", "wsp", "was", "wsd", "wad_come_from"]
tier1_table = variable_table.loc[TIER1_VARIABLES]

print()
print("=== Tier-1 modeled variables only ===")
print(tier1_table[["grid", "raw_bytes_per_snapshot", "decoded_bytes_per_snapshot"]])
print()
print(f"tier-1, raw on-disk, full window:       {tier1_table['raw_GB_full_archive'].sum():.1f} GB")
print(f"tier-1, decoded in-memory, full window: {tier1_table['decoded_GB_full_archive'].sum():.1f} GB "
      f"-- what train.ingest.load_training_window(..., TIER1_VARIABLES) over the FULL 4-year "
      f"window would hold in RAM at once, if loaded unchunked.")


=== Per-grid totals, full training window, ALL 16 variables ===
       raw_GB_full_archive  decoded_GB_full_archive
grid                                               
ocean                188.7                    905.6
wave                 177.7                    836.2
wind                  48.5                    291.2

ALL VARIABLES, raw on-disk:      414.9 GB
ALL VARIABLES, decoded in-memory: 2033.0 GB (2.03 TB) -- loading every variable decoded for the full window at once is not realistic RAM.

=== Tier-1 modeled variables only ===
               grid  raw_bytes_per_snapshot  decoded_bytes_per_snapshot
variable                                                               
sigwh          wave                 1788480                     7153920
wsh            wave                 1788480                     7153920
wsp            wave                 1788480                     7153920
was            wind                 2076480                     8305920
wsd            wave     

## 3. Summary

- **Grid sizes**: all 16 archive variables verified against `core.config`'s grid constants by actually round-tripping a synthetic snapshot through `core.io.snapshot_path`/`load_raw`/`load_snapshot` — see §1's `matches_table` column.
- **Memory footprint**: the full 4-year raw archive (all 16 variables) is several hundred GB on disk; decoding everything to `float64` at once would need on the order of 1-2 TB, clearly not something to hold in RAM simultaneously — real training code should load per-variable/windowed rather than materializing the whole decoded archive at once. The tier-1 modeled subset (6 variables) is far more tractable but still substantial for the full window — see §2's printed figures for exact numbers from this run.
- Re-run this notebook after any change to `core.config.VARIABLE_GRID`, the grid shape constants, or `core.io`'s continuous/direction classification — that's what it's for.